# Project - Airline AI Assistant (改良版)

改良重點：
- 改用 Ollama 本地模型 `llama3.1:8b`
- 新增 `set_ticket_price` tool，讓 LLM 可以修改票價
- 用 dispatch dict 取代 if/elif，讓 tool 擴充更容易
- 新增確認機制，避免誤操作寫入資料庫
- 清理重複定義的函式，只保留最終版本

In [1]:
import os
import json
import sqlite3
from openai import OpenAI
import gradio as gr

## 初始化 — 使用 Ollama 本地模型

In [2]:
# 使用 Ollama，確保本地已執行：ollama serve
# 並已下載模型：ollama pull llama3.1:8b

MODEL = "llama3.1:8b"
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'  # Ollama 不需要真實 API key
)

# 確認連線
try:
    models = client.models.list()
    print("✅ Ollama 連線成功")
    print("可用模型:", [m.id for m in models.data])
except Exception as e:
    print(f"❌ 無法連線到 Ollama: {e}")
    print("請確認 Ollama 正在執行（ollama serve）")

✅ Ollama 連線成功
可用模型: ['llama3.1:8b', 'gemma3:4b', 'gemma3:270m']


## 資料庫設定

In [3]:
DB = "prices.db"

def init_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
        conn.commit()

init_db()

# 預設票價資料
default_prices = {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999, "berlin": 499, "new york": 350}

def seed_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        for city, price in default_prices.items():
            cursor.execute(
                'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO NOTHING',
                (city, price)
            )
        conn.commit()

seed_db()
print("✅ 資料庫初始化完成")

✅ 資料庫初始化完成


## Tool 函式定義

In [4]:
def get_ticket_price(destination_city: str) -> str:
    """從資料庫查詢票價"""
    print(f"🔍 [Tool] 查詢票價: {destination_city}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
    if result:
        return f"Ticket price to {destination_city} is ${result[0]:.0f}"
    else:
        return f"No price data available for {destination_city}. Available cities: {', '.join(default_prices.keys())}"


def set_ticket_price(destination_city: str, new_price: float) -> str:
    """更新資料庫中的票價"""
    print(f"✏️ [Tool] 設定票價: {destination_city} -> ${new_price}")
    if new_price <= 0:
        return "Error: Price must be a positive number."
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',
            (destination_city.lower(), new_price, new_price)
        )
        conn.commit()
    return f"Ticket price for {destination_city} has been updated to ${new_price:.0f}."


# ✅ Dispatch dict：新增 tool 只需在這裡加一行
TOOL_FUNCTIONS = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price,
}

## Tool Schema 定義

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the price of a return ticket to the destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {
                        "type": "string",
                        "description": "The city the customer wants to travel to",
                    },
                },
                "required": ["destination_city"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_ticket_price",
            "description": "Set or update the ticket price for a destination city. Only use this when the user explicitly requests a price change and confirms it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {
                        "type": "string",
                        "description": "The city to update the ticket price for",
                    },
                    "new_price": {
                        "type": "number",
                        "description": "The new ticket price in USD",
                    },
                },
                "required": ["destination_city", "new_price"],
                "additionalProperties": False
            }
        }
    }
]

## System Message & Chat 邏輯

In [6]:
system_message = """
You are a helpful customer support assistant for FlightAI airline.
Give short, courteous answers, no more than 2 sentences.
Always be accurate. If you don't know the answer, say so.

You have access to tools to look up and update ticket prices.
IMPORTANT: Before calling set_ticket_price, always ask the user to confirm the change first.
Only proceed with the update after the user explicitly says yes or confirms.
"""

In [7]:
def handle_tool_calls(message) -> list:
    """使用 dispatch dict 處理所有 tool calls，支援多個 tool 同時呼叫"""
    responses = []
    for tool_call in message.tool_calls:
        fn_name = tool_call.function.name
        fn = TOOL_FUNCTIONS.get(fn_name)

        if fn is None:
            content = f"Error: Unknown tool '{fn_name}'"
        else:
            arguments = json.loads(tool_call.function.arguments)
            content = fn(**arguments)

        responses.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        })
    return responses


def chat(message, history):
    """主要 chat 函式，支援連續 tool call（agentic loop）"""
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # Agentic loop：持續處理 tool calls 直到模型給出最終回覆
    max_iterations = 5  # 防止無限迴圈
    iterations = 0
    while response.choices[0].finish_reason == "tool_calls" and iterations < max_iterations:
        tool_message = response.choices[0].message
        tool_responses = handle_tool_calls(tool_message)
        messages.append(tool_message)
        messages.extend(tool_responses)
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        iterations += 1

    return response.choices[0].message.content

## 快速測試（不啟動 UI）

In [8]:
# 直接測試工具函式
print(get_ticket_price("Tokyo"))
print(get_ticket_price("Sydney"))
print(get_ticket_price("Miami"))  # 不存在的城市

set_ticket_price("miami", 650)
print(get_ticket_price("Miami"))  # 新增後再查詢

🔍 [Tool] 查詢票價: Tokyo
Ticket price to Tokyo is $1420
🔍 [Tool] 查詢票價: Sydney
Ticket price to Sydney is $2999
🔍 [Tool] 查詢票價: Miami
No price data available for Miami. Available cities: london, paris, tokyo, sydney, berlin, new york
✏️ [Tool] 設定票價: miami -> $650
🔍 [Tool] 查詢票價: Miami
Ticket price to Miami is $650


## 啟動 Gradio UI

In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="✈️ FlightAI Customer Support",
    description="Ask about ticket prices or request price updates. Powered by llama3.1:8b via Ollama.",
    examples=[
        "How much is a ticket to Tokyo?",
        "What's the cheapest destination you have?",
        "Can you update the price for London to $850?",
    ]
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


🔍 [Tool] 查詢票價: none
🔍 [Tool] 查詢票價: 
